In [7]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [8]:
# Constants

MODEL = "llama3.2"

In [9]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [22]:
# Let's try one out

ed = Website("https://playvalorant.com/en-gb/")
print(ed.title)
print(ed.text)

VALORANT
VALORANT — A 5v5 tactical shooter featuring agents with unique abilities
Play For Free
THE LATEST
GO TO NEWS PAGE
Esport
2026-06-25T14:00:00.000Z
2026 VCT EMEA Stage 2
Challenger teams join the competition on the road to Barcelona Finals!
Game Updates
2026-06-23T15:00:00.000Z
Summit Gameplay Reveal Trailer - VALORANT
See the peak of beauty in the battlefield. Only on Summit.
Game Updates
2026-06-23T13:00:00.000Z
VALORANT Patch Notes 13.00
Act 4 starts now.
SEASON 26: ACT IV – LIVE NOW
Experience the Summit Map, new Retake game mode, Blackspyre skins, and a chance to reach your peak.
WATCH NOW
WE ARE VALORANT
DEFY THE LIMITS
Blend your style and experience on a global, competitive stage. You have 13 rounds to attack and defend your side using sharp gunplay and tactical abilities. And, with one life per-round, you'll need to think faster than your opponent if you want to survive. Take on foes across Competitive and Unranked modes as well as Deathmatch and Spike Rush.
LEARN THE G

In [13]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [14]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "The contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [15]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [23]:
# And now: call the Ollama function instead of OpenAI

def summarize(url):
    website = Website("https://playvalorant.com/en-gb/")
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']

In [24]:
summarize("https://playvalorant.com/en-gb/")

'**VALORANT Website Summary**\n==========================\n\n### Overview\n\nVALORANT is a 5v5 tactical shooter game that features agents with unique abilities.\n\n### Latest News and Announcements\n\n* **2026-06-25T14:00:00.000Z**: The 2026 VCT EMEA Stage 2 has begun, featuring challenger teams competing on the road to Barcelona Finals.\n* **2026-06-23T15:00:00.000Z**: A new gameplay reveal trailer for Summit was released, showcasing the peak of beauty in the battlefield.\n* **2026-06-23T13:00:00.000Z**: VALORANT Patch Notes 13.00 were released, marking the start of Act 4 and introducing a new Retake game mode.\n\n### Game Features\n\n* **Play For Free**\n* **Esport** with Competitive and Unranked modes\n* **Game Modes**: Deathmatch, Spike Rush, and more\n* **Agents**: With unique abilities that create opportunities for creative gameplay\n* **Maps**: Each map is a playground to showcase team strategies and spectacular plays\n\n### Seasonal Updates\n\n* **SEASON 26: ACT IV – LIVE NOW**

In [18]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [25]:
display_summary("https://playvalorant.com/en-gb/")

**VALORANT Game Overview**
=====================================

*   **Game Type:** 5v5 tactical shooter
*   **Unique Features:** Agents with unique abilities, sharp gunplay, and tactical abilities

**News and Announcements**
---------------------------

*   **2026-06-25T14:00:00.000Z**: 2026 VCT EMEA Stage 2 - Challenger teams join the competition on the road to Barcelona Finals.
*   **2026-06-23T15:00:00.000Z**: Summit Gameplay Reveal Trailer - VALORANT
*   **2026-06-23T13:00:00.000Z**: VALORANT Patch Notes 13.00 - Act 4 starts now, featuring the Summit Map, new Retake game mode, Blackspyre skins, and more.

**Game Modes**
----------------

*   **Competitive Mode:** Attack and defend your side using sharp gunplay and tactical abilities.
*   **Unranked Mode:** Play without ranking restrictions.
*   **Deathmatch Mode:** Fast-paced, high-scoring gameplay.
*   **Spike Rush Mode:** A fast-paced, 5v5 game mode with a twist.

**Agents**
------------

*   **Unique Abilities:** Each Agent has adaptive, swift, and lethal abilities that create opportunities for gunplay.
*   **No Two Agents Play Alike:** Unique playstyles and highlight reels.

**Maps**
---------

*   **Purpose-Built Maps:** Spectacular plays and clutch moments await on each map.